In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [29]:
def cmp(s, dt, t):
    ex = torch.equal(dt, t.grad)
    app = torch.allclose(dt, t.grad)
    max_diff = (t.grad - dt).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {max_diff}')

In [7]:
n_embd = 10
n_hidden = 64
g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

4137


In [8]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [24]:
# forward pass


emb = C[Xb]
embcat = emb.view(emb.shape[0],-1)
# linear layer 1
hprebn = emb_cat @ W1 +b1

# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias


# Non-linearity
h = torch.tanh(hpreact) # hidden layer


# Linear layer 2
logits = h @ W2 + b2

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

In [25]:
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3477, grad_fn=<NegBackward0>)

In [62]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# -----------------


# dlogprobs

# the loss is defined as taking negative and indexing the true labels and finally taking the mean
# so the differential is just 1 for the element which is a true label and 0 otherwise

dlogprobs = torch.zeros(logprobs.shape)
# and then for specific index make it 1/n since we take mean
dlogprobs[range(n),Yb]=-1.0/n

cmp('logprobs', dlogprobs, logprobs)

# probs

# dloss/dprobs = dloss/dlogprobs * dlogprobs/dprobs

# dlogprobs/dprobs = 1/probs

# and dloss/dlogprobs is already stored in dlogprobs both have same dimesion we just have to do element wise multiplicaiton
dprobs = dlogprobs * 1/probs
cmp('probs', dprobs, probs)



dcounts_sum_inv = (dprobs * counts).sum(1,keepdims=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)




dcounts_sum = dcounts_sum_inv * (-counts_sum**-2)
cmp('counts_sum', dcounts_sum, counts_sum)

dcounts = dprobs * counts_sum_inv
dcounts += dcounts_sum.expand_as(counts)
cmp('counts', dcounts, counts)


dnorm_logits = dcounts * counts
cmp('norm_logits', dnorm_logits, norm_logits)


dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)


dlogits = dnorm_logits.clone()
for i in range(n):

    # find index of maximum
    mx = 0
    for j in range(vocab_size):
        if logits[i,j] > logits[i,mx]:
            mx = j

    dlogits[i,mx] += dlogit_maxes[i].item()

cmp('logits', dlogits, logits)

dh = dlogits @ W2.T
cmp('h', dh, h)



dW2 = h.T @ dlogits
cmp('W2', dW2, W2)

db2 = dlogits.sum(0)
cmp('b2', db2, b2)

dhpreact = dh * (1 - h**2)
cmp('hpreact', dhpreact, hpreact)

dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
cmp('bngain', dbngain, bngain)
dbnbias = dhpreact.sum(0, keepdim=True)
cmp('bnbias', dbnbias, bnbias)
dbnraw = dhpreact * bngain
cmp('bnraw', dbnraw, bnraw)

dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)

dbnvar = dbnvar_inv * (-0.5) * (bnvar + 1e-5)**(-1.5)
cmp('bnvar', dbnvar, bnvar)
dbndiff2 = dbnvar.expand_as(bndiff2) / (n - 1)
cmp('bndiff2', dbndiff2, bndiff2)

dbndiff = dbnraw * bnvar_inv
dbndiff += dbndiff2 * 2 * bndiff
cmp('bndiff', dbndiff, bndiff)
dbnmeani = -dbndiff.sum(0, keepdim=True)
cmp('bnmeani', dbnmeani, bnmeani)
dhprebn = dbndiff.clone()
dhprebn += dbnmeani.expand_as(hprebn) / n
cmp('hprebn', dhprebn, hprebn)



dembcat = dhprebn @ W1.T
#cmp('embcat', dembcat, embcat)


dW1 = embcat.T @ dhprebn
cmp('W1', dW1, W1)
db1 = dhprebn.sum(0)
cmp('b1', db1, b1)
demb = dembcat.view_as(emb)
#cmp('emb', demb, emb)

dC = torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        dC[Xb[i, j]] += demb[i, j]
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10
bngain          | exact: False | approximate: True  | maxdiff: 1.3969838619232178e-09
bnbias          | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09
bnraw  

In [64]:
dlogits = F. softmax (logits, 1)
dlogits [range(n), Yb] -= 1
dlogits /= n
cmp ('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 5.3551048040390015e-09


In [ ]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

In [65]:
dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw) . sum(0))
cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


In [68]:
import torch
import torch.nn.functional as F

n_embd = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
bngain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = False

# 2. Training loop
max_steps = 10000
batch_size = 32
n = batch_size
learning_rate = 0.1

print(f"Starting manual backprop training loop for {max_steps} steps...")
print("-" * 50)

for step in range(max_steps):

    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)

    # Linear Layer 1
    hprebn = embcat @ W1 + b1

    # BatchNorm Layer
    bnmeani = 1/n * hprebn.sum(0, keepdim=True)
    bndiff = hprebn - bnmeani
    bndiff2 = bndiff**2
    bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = bndiff * bnvar_inv
    hpreact = bngain * bnraw + bnbias

    # Activation
    h = torch.tanh(hpreact)

    # Linear Layer 2
    logits = h @ W2 + b2

    # Track cross-entropy loss
    loss = F.cross_entropy(logits, Yb)

    # --- Manual Backward Pass (Putting it all together) ---

    dlogits = F.softmax(logits, 1)
    dlogits[range(n), Yb] -= 1
    dlogits /= n

    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)
    dh = dlogits @ W2.T
    dhpreact = dh * (1 - h**2)

    dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)
    dhprebn = bngain * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0) - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0))

    # 4. Backprop into Layer 1 weights & concatenated embedding
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)
    dembcat = dhprebn @ W1.T
    demb = dembcat.view_as(emb)

    # Backprop into Embedding matrix lookup
    dC = torch.zeros_like(C)
    for i in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            dC[Xb[i, j]] += demb[i, j]

    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    for p, grad in zip(parameters, grads):
        p.data -= learning_rate * grad

    if step % 1000 == 0 or step == max_steps - 1:
        print(f"Step {step:4d}/{max_steps} | Loss: {loss.item():.4f}")

Starting manual backprop training loop for 10000 steps...
--------------------------------------------------
Step    0/10000 | Loss: 3.3211
Step 1000/10000 | Loss: 2.1373
Step 2000/10000 | Loss: 2.5454
Step 3000/10000 | Loss: 2.5532
Step 4000/10000 | Loss: 2.3102
Step 5000/10000 | Loss: 2.3552
Step 6000/10000 | Loss: 2.1411
Step 7000/10000 | Loss: 2.3702
Step 8000/10000 | Loss: 2.1965
Step 9000/10000 | Loss: 2.2849
Step 9999/10000 | Loss: 2.3088


In [70]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmeani) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))

carlah.
amille.
khi.
mrix.
thty.
salanskeer.
hutefaherric.
kaqeig.
elmarahceriiv.
kaleig.
dhan.
join.
qhinthanlin.
alian.
quwa.
elon.
jarixi.
jaheeniruli.
edde.
iia.


##Wavenet

In [72]:
# build the dataset
block_size = 8 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

torch.Size([182625, 8]) torch.Size([182625])
torch.Size([22655, 8]) torch.Size([22655])
torch.Size([22866, 8]) torch.Size([22866])
........ --> y
.......y --> u
......yu --> h
.....yuh --> e
....yuhe --> n
...yuhen --> g
..yuheng --> .
........ --> d
.......d --> i
......di --> o
.....dio --> n
....dion --> d
...diond --> r
..diondr --> e
.diondre --> .
........ --> x
.......x --> a
......xa --> v
.....xav --> i
....xavi --> e


In [73]:
# Near copy paste of the layers we have developed in Part 3 (copied form karpathy's notebook)

# -----------------------------------------------------------------------------------------------
class Linear:

  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5 # note: kaiming init
    self.bias = torch.zeros(fan_out) if bias else None

  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out

  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])

# -----------------------------------------------------------------------------------------------
class BatchNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)

  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      if x.ndim == 2:
        dim = 0
      elif x.ndim == 3:
        dim = (0,1)
      xmean = x.mean(dim, keepdim=True) # batch mean
      xvar = x.var(dim, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

# -----------------------------------------------------------------------------------------------
class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

# -----------------------------------------------------------------------------------------------
class Embedding:

  def __init__(self, num_embeddings, embedding_dim):
    self.weight = torch.randn((num_embeddings, embedding_dim))

  def __call__(self, IX):
    self.out = self.weight[IX]
    return self.out

  def parameters(self):
    return [self.weight]

# -----------------------------------------------------------------------------------------------
class FlattenConsecutive:

  def __init__(self, n):
    self.n = n

  def __call__(self, x):
    B, T, C = x.shape
    x = x.view(B, T//self.n, C*self.n)
    if x.shape[1] == 1:
      x = x.squeeze(1)
    self.out = x
    return self.out

  def parameters(self):
    return []

# -----------------------------------------------------------------------------------------------
class Sequential:

  def __init__(self, layers):
    self.layers = layers

  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    self.out = x
    return self.out

  def parameters(self):
    # get parameters of all layers and stretch them out into one list
    return [p for layer in self.layers for p in layer.parameters()]


In [75]:
import torch
import torch.nn.functional as F

vocab_size = 27
n_embd = 24
n_hidden = 128

model = Sequential([
  Embedding(vocab_size, n_embd),

  FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),

  Linear(n_hidden, vocab_size)
])

parameters = model.parameters()
print(f"Total number of parameters: {sum(p.nelement() for p in parameters)}")
for p in parameters:
  p.requires_grad = True

max_steps = 200000
batch_size = 32
loss_log = []


for i in range(max_steps):

  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix] # Batch X and Y

  for layer in model.layers:
    if isinstance(layer, BatchNorm1d):
      layer.training = True

  logits = model(Xb)
  loss = F.cross_entropy(logits, Yb)

  for p in parameters:
    p.grad = None
  loss.backward()

  lr = 0.1 if i < 150000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad


  if i % 10000 == 0:
    print(f"Step {i:7d}/{max_steps:7d} | Loss: {loss.item():.4f}")
  loss_log.append(loss.log10().item())

print("Training complete!")


@torch.no_grad()
def evaluate_split(X_split, Y_split):
  for layer in model.layers:
    if isinstance(layer, BatchNorm1d):
      layer.training = False

  logits = model(X_split)
  loss = F.cross_entropy(logits, Y_split)
  return loss.item()

train_loss = evaluate_split(Xtr, Ytr)
dev_loss   = evaluate_split(Xdev, Ydev)
test_loss  = evaluate_split(Xte, Yte)

print("\n" + "="*35)
print(f"  FINAL MODEL METRICS")
print("="*35)
print(f"  Training Loss:   {train_loss:.4f}")
print(f"  Validation Loss: {dev_loss:.4f}")
print(f"  Test Loss:       {test_loss:.4f}")
print("="*35)

Total number of parameters: 76579
Step       0/ 200000 | Loss: 3.5027
Step   10000/ 200000 | Loss: 1.6629
Step   20000/ 200000 | Loss: 1.9045
Step   30000/ 200000 | Loss: 1.8133
Step   40000/ 200000 | Loss: 1.9177
Step   50000/ 200000 | Loss: 2.2145
Step   60000/ 200000 | Loss: 1.6658
Step   70000/ 200000 | Loss: 1.9602
Step   80000/ 200000 | Loss: 1.6630
Step   90000/ 200000 | Loss: 1.6884
Step  100000/ 200000 | Loss: 1.9036
Step  110000/ 200000 | Loss: 1.8776
Step  120000/ 200000 | Loss: 2.6358
Step  130000/ 200000 | Loss: 1.5128
Step  140000/ 200000 | Loss: 1.9015
Step  150000/ 200000 | Loss: 1.8864
Step  160000/ 200000 | Loss: 1.7736
Step  170000/ 200000 | Loss: 1.9279
Step  180000/ 200000 | Loss: 1.9027
Step  190000/ 200000 | Loss: 1.8907
Training complete!

  FINAL MODEL METRICS
  Training Loss:   1.7658
  Validation Loss: 1.9907
  Test Loss:       1.9836
